# 03 · Clustering Demo
**Epidemiological Pulse – Population Health Hotspot Detection**

This notebook demonstrates the full HDBSCAN hotspot detection pipeline:
1. Aggregate features to per-ZIP time windows
2. Run HDBSCAN with geospatial weighting
3. Assign risk levels
4. Visualise clusters on an interactive map
5. Export GeoJSON
6. Run Prophet forecasting per cluster


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import json
from pathlib import Path

import yaml
with open('../config/params.yaml') as f:
    config = yaml.safe_load(f)

from src.data.preprocess import PreprocessingPipeline
from src.features.build_features import FeatureEngineeringPipeline
from src.models.cluster_hotspots import HotspotDetectionPipeline

DATA_DIR = Path('../data/synthetic')
OUT_DIR  = Path('../outputs')
print('Ready.')

## 1 · Prepare Feature Matrix

In [ ]:
raw = {
    'pharmacy':   pd.read_csv(DATA_DIR / 'pharmacy_sales.csv',        parse_dates=['date']),
    'sentiment':  pd.read_csv(DATA_DIR / 'social_media_sentiment.csv', parse_dates=['date']),
    'weather':    pd.read_csv(DATA_DIR / 'weather.csv',               parse_dates=['date']),
    'aqi':        pd.read_csv(DATA_DIR / 'aqi.csv',                   parse_dates=['date']),
    'er_visits':  pd.read_csv(DATA_DIR / 'er_visits.csv',             parse_dates=['date']),
}

preprocess_pipeline = PreprocessingPipeline(config)
df_clean, *_ = preprocess_pipeline.run(raw)

fe_pipeline = FeatureEngineeringPipeline(config)
df_features = fe_pipeline.run(df_clean)

print(f'Feature matrix: {df_features.shape}')

## 2 · Run HDBSCAN Hotspot Detection

In [ ]:
hotspot_pipeline = HotspotDetectionPipeline(config)
results = hotspot_pipeline.run(df_features)

hotspots   = results['hotspots']      # per-ZIP aggregated results with cluster labels
geojson    = results['geojson']       # GeoJSON FeatureCollection
clusterer  = results['clusterer']     # fitted HDBSCANClusterer instance

print('Hotspot summary:')
display(hotspots[['zip_code', 'cluster_label', 'risk_level', 'risk_score']].head(20))

## 3 · Cluster Distribution

In [ ]:
risk_counts = hotspots['risk_level'].value_counts().reset_index()
risk_counts.columns = ['risk_level', 'count']

color_map = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

fig = px.bar(
    risk_counts,
    x='risk_level', y='count',
    color='risk_level',
    color_discrete_map=color_map,
    title='ZIP Codes by Risk Level',
    template='plotly_dark'
)
fig.show()

# Noise points (cluster_label == -1 means HDBSCAN found no cluster)
noise = (hotspots['cluster_label'] == -1).sum()
print(f'Noise points (unclustered ZIPs): {noise}')

## 4 · Interactive Hotspot Map

In [ ]:
if 'latitude' in hotspots.columns and 'longitude' in hotspots.columns:
    fig = px.scatter_mapbox(
        hotspots,
        lat='latitude',
        lon='longitude',
        color='risk_level',
        size='risk_score',
        size_max=30,
        color_discrete_map=color_map,
        hover_name='zip_code',
        hover_data={'risk_score': ':.3f', 'cluster_label': True},
        mapbox_style='carto-darkmatter',
        zoom=3,
        center={'lat': 39.5, 'lon': -98.35},
        title='Disease Outbreak Hotspots by ZIP Code',
        template='plotly_dark',
        height=600
    )
    fig.show()
else:
    print('Latitude/longitude columns not found in hotspots DataFrame.')

## 5 · GeoJSON Inspection

In [ ]:
if geojson:
    print(f'GeoJSON type: {geojson["type"]}')
    print(f'Features:     {len(geojson["features"])}')
    print('\nFirst feature properties:')
    print(json.dumps(geojson['features'][0]['properties'], indent=2))
else:
    print('No GeoJSON generated.')

## 6 · High-Risk ZIP Deep Dive

In [ ]:
high_risk = hotspots[hotspots['risk_level'] == 'High']

if len(high_risk) == 0:
    print('No high-risk ZIPs found – try increasing outbreak spike magnitude in config.')
else:
    print(f'High-risk ZIP codes: {high_risk["zip_code"].tolist()}')

    # Plot ER visits for highest-risk ZIP
    top_zip = high_risk.sort_values('risk_score', ascending=False).iloc[0]['zip_code']
    df_top = df_features[df_features['zip_code'] == top_zip].sort_values('date')

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_top['date'], y=df_top['er_visits'],
        name='ER Visits', line=dict(color='red')
    ))
    if 'pharmacy_sales' in df_top.columns:
        fig.add_trace(go.Scatter(
            x=df_top['date'], y=df_top['pharmacy_sales'],
            name='Pharmacy Sales', yaxis='y2',
            line=dict(color='steelblue', dash='dot')
        ))

    fig.update_layout(
        title=f'Highest Risk ZIP: {top_zip}',
        yaxis2=dict(overlaying='y', side='right', title='Pharmacy Sales'),
        template='plotly_dark'
    )
    fig.show()

print('\n✅ Clustering demo complete.')